In [ ]:
!pip install clifford -q

In [ ]:
"""Independent Geometric Algebra verification for the known pose VAE paper.

The neural models in the paper implement the induced rotation matrices because
PyTorch's grid samplers consume affine coordinate maps. This script uses the
Python `clifford` package as an independent algebraic check that the planar
and spatial rotor sandwich products induce the same matrices used in the code.

It verifies:
  1. 2D rotor unit norm, vector sandwich action, and composition.
  2. 3D spatial rotor/unit-quaternion unit norm, vector sandwich action and
     composition for the same quaternion to SO(3) convention used by the voxel
     experiment.
"""
from __future__ import annotations

import json
import math
import platform
from typing import Dict, Tuple

import numpy as np

try:
    import clifford
    from clifford import Cl
except Exception as exc:  # pragma: no cover
    raise SystemExit(
        "This script requires the Python package 'clifford'. Install with: "
        "pip install clifford\n"
        f"Original import error: {exc}"
    )


def active_matrix_2d(theta: float) -> np.ndarray:
    c = math.cos(theta)
    s = math.sin(theta)
    return np.array([[c, -s], [s, c]], dtype=np.float64)


def quat_to_matrix(q: np.ndarray) -> np.ndarray:
    """Quaternion [w, x, y, z] to active SO(3) matrix.

    This is the same convention used in known_pose_experiment_code.py.
    """
    w, x, y, z = [float(v) for v in q]
    n = math.sqrt(w * w + x * x + y * y + z * z)
    w, x, y, z = w / n, x / n, y / n, z / n
    return np.array(
        [
            [1 - 2 * y * y - 2 * z * z, 2 * x * y - 2 * z * w, 2 * x * z + 2 * y * w],
            [2 * x * y + 2 * z * w, 1 - 2 * x * x - 2 * z * z, 2 * y * z - 2 * x * w],
            [2 * x * z - 2 * y * w, 2 * y * z + 2 * x * w, 1 - 2 * x * x - 2 * y * y],
        ],
        dtype=np.float64,
    )


def quat_multiply(q2: np.ndarray, q1: np.ndarray) -> np.ndarray:
    """Hamilton product q2*q1 for active composition R(q2)R(q1)."""
    w2, x2, y2, z2 = [float(v) for v in q2]
    w1, x1, y1, z1 = [float(v) for v in q1]
    out = np.array(
        [
            w2 * w1 - x2 * x1 - y2 * y1 - z2 * z1,
            w2 * x1 + x2 * w1 + y2 * z1 - z2 * y1,
            w2 * y1 - x2 * z1 + y2 * w1 + z2 * x1,
            w2 * z1 + x2 * y1 - y2 * x1 + z2 * w1,
        ],
        dtype=np.float64,
    )
    return out / np.linalg.norm(out)


def scalar_part(mv) -> float:
    """Return scalar coefficient of a clifford multivector."""
    return float(mv[()])


def verify_2d(rng: np.random.Generator, n: int) -> Dict[str, float]:
    _layout, blades = Cl(2)
    e1, e2 = blades["e1"], blades["e2"]
    I = e1 ^ e2

    max_unit_norm_error = 0.0
    max_vector_action_error = 0.0
    max_composition_action_error = 0.0
    max_composition_matrix_error = 0.0

    for _ in range(n):
        theta = float(rng.uniform(-math.pi, math.pi))
        x = rng.normal(size=2)
        c = math.cos(theta / 2.0)
        s = math.sin(theta / 2.0)
        r = c - s * I
        u = float(x[0]) * e1 + float(x[1]) * e2

        max_unit_norm_error = max(max_unit_norm_error, abs(scalar_part(r * ~r) - 1.0))

        y_mv = r * u * ~r
        y_clifford = np.array([float((y_mv | e1)[()]), float((y_mv | e2)[()])], dtype=np.float64)
        y_matrix = active_matrix_2d(theta) @ x
        max_vector_action_error = max(max_vector_action_error, float(np.max(np.abs(y_clifford - y_matrix))))

        theta1 = float(rng.uniform(-math.pi, math.pi))
        theta2 = float(rng.uniform(-math.pi, math.pi))
        r1 = math.cos(theta1 / 2.0) - math.sin(theta1 / 2.0) * I
        r2 = math.cos(theta2 / 2.0) - math.sin(theta2 / 2.0) * I
        r12 = math.cos((theta1 + theta2) / 2.0) - math.sin((theta1 + theta2) / 2.0) * I
        x2 = rng.normal(size=2)
        u2 = float(x2[0]) * e1 + float(x2[1]) * e2
        y_prod = (r2 * r1) * u2 * ~(r2 * r1)
        y_sum = r12 * u2 * ~r12
        y_prod_vec = np.array([float((y_prod | e1)[()]), float((y_prod | e2)[()])], dtype=np.float64)
        y_sum_vec = np.array([float((y_sum | e1)[()]), float((y_sum | e2)[()])], dtype=np.float64)
        max_composition_action_error = max(
            max_composition_action_error, float(np.max(np.abs(y_prod_vec - y_sum_vec)))
        )
        max_composition_matrix_error = max(
            max_composition_matrix_error,
            float(np.max(np.abs(active_matrix_2d(theta2) @ active_matrix_2d(theta1) - active_matrix_2d(theta1 + theta2)))),
        )

    return {
        "max_unit_norm_error": max_unit_norm_error,
        "max_vector_action_error": max_vector_action_error,
        "max_composition_action_error": max_composition_action_error,
        "max_composition_matrix_error": max_composition_matrix_error,
    }


def verify_3d(rng: np.random.Generator, n: int) -> Dict[str, float]:
    _layout, blades = Cl(3)
    e1, e2, e3 = blades["e1"], blades["e2"], blades["e3"]
    e23 = e2 ^ e3
    e31 = e3 ^ e1
    e12 = e1 ^ e2

    def rotor_from_quat(q: np.ndarray):
        w, x, y, z = [float(v) for v in q]
        return w - x * e23 - y * e31 - z * e12

    max_unit_norm_error = 0.0
    max_vector_action_error = 0.0
    max_composition_action_error = 0.0
    max_composition_matrix_error = 0.0

    for _ in range(n):
        q = rng.normal(size=4)
        q = q / np.linalg.norm(q)
        x = rng.normal(size=3)
        r = rotor_from_quat(q)
        u = float(x[0]) * e1 + float(x[1]) * e2 + float(x[2]) * e3

        max_unit_norm_error = max(max_unit_norm_error, abs(scalar_part(r * ~r) - 1.0))

        y_mv = r * u * ~r
        y_clifford = np.array(
            [float((y_mv | e1)[()]), float((y_mv | e2)[()]), float((y_mv | e3)[()])],
            dtype=np.float64,
        )
        y_matrix = quat_to_matrix(q) @ x
        max_vector_action_error = max(max_vector_action_error, float(np.max(np.abs(y_clifford - y_matrix))))

        q1 = rng.normal(size=4); q1 = q1 / np.linalg.norm(q1)
        q2 = rng.normal(size=4); q2 = q2 / np.linalg.norm(q2)
        r1 = rotor_from_quat(q1)
        r2 = rotor_from_quat(q2)
        q21 = quat_multiply(q2, q1)
        r21 = rotor_from_quat(q21)
        x2 = rng.normal(size=3)
        u2 = float(x2[0]) * e1 + float(x2[1]) * e2 + float(x2[2]) * e3
        y_prod = (r2 * r1) * u2 * ~(r2 * r1)
        y_qmul = r21 * u2 * ~r21
        y_prod_vec = np.array(
            [float((y_prod | e1)[()]), float((y_prod | e2)[()]), float((y_prod | e3)[()])],
            dtype=np.float64,
        )
        y_qmul_vec = np.array(
            [float((y_qmul | e1)[()]), float((y_qmul | e2)[()]), float((y_qmul | e3)[()])],
            dtype=np.float64,
        )
        max_composition_action_error = max(
            max_composition_action_error, float(np.max(np.abs(y_prod_vec - y_qmul_vec)))
        )
        max_composition_matrix_error = max(
            max_composition_matrix_error,
            float(np.max(np.abs(quat_to_matrix(q2) @ quat_to_matrix(q1) - quat_to_matrix(q21)))),
        )

    return {
        "max_unit_norm_error": max_unit_norm_error,
        "max_vector_action_error": max_vector_action_error,
        "max_composition_action_error": max_composition_action_error,
        "max_composition_matrix_error": max_composition_matrix_error,
    }


def main() -> None:
    seed = 12345
    n = 1000
    rng = np.random.default_rng(seed)
    report: Dict[str, object] = {
        "software": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "clifford": getattr(clifford, "__version__", "unknown"),
        },
        "seed": seed,
        "num_random_tests_per_block": n,
        "two_dimensional_rotor": verify_2d(rng, n),
        "three_dimensional_spatial_rotor": verify_3d(rng, n),
    }
    print(json.dumps(report, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()


{
  "num_random_tests_per_block": 1000,
  "seed": 12345,
  "software": {
    "clifford": "1.5.1",
    "numpy": "2.0.2",
    "python": "3.12.13"
  },
  "three_dimensional_spatial_rotor": {
    "max_composition_action_error": 1.7763568394002505e-15,
    "max_composition_matrix_error": 8.881784197001252e-16,
    "max_unit_norm_error": 4.440892098500626e-16,
    "max_vector_action_error": 1.3322676295501878e-15
  },
  "two_dimensional_rotor": {
    "max_composition_action_error": 1.2212453270876722e-15,
    "max_composition_matrix_error": 5.134781488891349e-16,
    "max_unit_norm_error": 2.220446049250313e-16,
    "max_vector_action_error": 8.881784197001252e-16
  }
}
